**Cell #01**

# RAG11 Nutrition — Stage 2: Ask Sample Questions

Runs a small set of hand-picked nutrition questions end to end through the
retrieval + generation pipeline built in Stage 1:

1. Embed the question with Voyage AI (`input_type="query"`, the counterpart
   to the `input_type="document"` used when the child chunks themselves were
   embedded in `stage1_2_eda_load_chunks.ipynb` -- Voyage's embeddings are
   asymmetric, tuned differently for the query side vs. the document side).
2. Retrieve the best-matching child chunks via the `match_rag11_child_chunks`
   RPC (`sql/create_sql_tables.sql`). At least 3 chunks always go to the LLM,
   so no answer is ever generated from a single, possibly-unlucky match.
3. Ask Claude to answer strictly from those retrieved excerpts. When a
   question has a clean Yes/No answer, the reply leads with a
   `Short answer: Yes` / `Short answer: No` line before the full
   explanation; open-ended questions just get the full explanation.
4. Every answer also reports the source page numbers behind it (parsed from
   each retrieved chunk's `[Source: ... | Pages X-Y]` contextual header), so
   an answer can always be checked against the original PDF.

**Before running this notebook**: `stage1_2_eda_load_chunks.ipynb` must
already have loaded and embedded your chunks into Supabase (its own
prerequisite is `sql/create_sql_tables.sql`, including the
`match_rag11_child_chunks` RPC).

**2026-09-17 update:** retrieval/generation logic now lives in `./reusable_code/` (shared with `stage2_ask_examples2_rerank.ipynb`, which adds an optional reranking pass on top of this same pipeline) -- see `reusable_code/README.md`.


In [1]:
# Cell #02
from reusable_code import init_clients, EMBEDDING_MODEL, GENERATION_MODEL
from reusable_code.env import optional_env

# Client construction (Supabase / Voyage / Anthropic) and env-var loading now
# live in ./reusable_code/clients.py, shared with stage2_ask_examples2_rerank.ipynb
# -- see reusable_code/README.md for the full guide. EMBEDDING_MODEL and
# GENERATION_MODEL are defined there too now (must match
# stage1_2_eda_load_chunks.ipynb's EMBEDDING_MODEL).
clients = init_clients()
supabase = clients.supabase
voyage_client = clients.voyage
anthropic_client = clients.anthropic

print("Clients ready. Supabase project:", optional_env("PUBLIC_SUPABASE_URL"))


Clients ready. Supabase project: https://czgrxgzdmodkkmbmraub.supabase.co


**Cell #03**

## Retrieval

In [2]:
# Cell #04
from reusable_code import (
    with_retry as _retry,
    MIN_CONTEXT_CHUNKS,
    NUM_CONTEXT_CHUNKS,
    embed_query,
    retrieve_chunks,
    page_numbers_for_chunk,
)

# embed_query() / retrieve_chunks() / page_numbers_for_chunk() now live in
# ./reusable_code/retrieval.py (shared with stage2_ask_examples2_rerank.ipynb).
# They use the `clients` bundle from the cell above by default, so calls
# below are unchanged: retrieve_chunks(question, match_count=...).


**Cell #05**

## Generation

In [3]:
# Cell #06
from reusable_code import (
    SYSTEM_PROMPT,
    MAX_ANSWER_TOKENS,
    build_context_block,
    extract_short_answer,
    grounding_words,
    ask_question,
)

# build_context_block() / extract_short_answer() / grounding_words() /
# ask_question() now live in ./reusable_code/generation.py (shared with
# stage2_ask_examples2_rerank.ipynb). ask_question() also gained optional
# `use_hybrid` / `use_hyde` / `use_multi_query` / `use_rerank` /
# `expand_to_parents` keywords; the first four default to whatever
# reusable_code/config.py reads from .env (True unless you set them to
# False there). This notebook passes them explicitly as False below so it
# keeps demonstrating plain vector search -- this notebook's original
# behavior -- regardless of what .env is set to. See
# stage2_ask_examples2_rerank.ipynb onward for worked examples of each
# technique, or reusable_code/README.md for the full guide.


**Cell #07**

## The 5 questions

A mix on purpose: some have a clean Yes/No answer (to exercise the
`Short answer:` formatting), some are open-ended (to check that format is
correctly skipped), and two are phrased as common oversimplifications so a
good answer has to lean on the nuance actually present in the source text
rather than a flat "yes"/"no".

In [4]:
# Cell #08
SAMPLE_QUESTIONS = [
    # Yes/No -- a clean textbook fact, good for checking the Short answer format.
    "Is vitamin C a water-soluble vitamin?",
    # Yes/No, but the honest answer is nuanced -- checks the model still
    # commits to Yes/No first and puts the nuance in the full explanation.
    "Does eating excess dietary protein get stored directly as body fat?",
    # Open-ended, multipart -- likely needs chunks from more than one source.
    "What roles do carbohydrates, proteins, and fats each play in providing energy to the body?",
    # Yes/No, commonly oversimplified -- tests whether retrieval surfaces the
    # nuance rather than a flat "yes, all saturated fat is bad".
    "Is saturated fat the only type of dietary fat linked to raised LDL cholesterol?",
    # Open-ended -- no yes/no framing at all.
    "What functions does dietary fiber serve in the digestive system?",
]

**Cell #09**

## Run all 5 questions

In [5]:
# Cell #10
results = []
for question in SAMPLE_QUESTIONS:
    print(f"Q: {question}")
    result = ask_question(
        question,
        use_hybrid=False, use_hyde=False, use_multi_query=False, expand_to_parents=False,
    )
    results.append(result)

    print(f"  chunks used: {result['chunks_used']} (source(s): {', '.join(result['source_keys'])})")
    print(f"  source pages: {result['source_pages']}")
    if result["short_answer"]:
        print(f"  Short answer: {result['short_answer']}")
    print()
    print(result["answer"])
    print()
    grounding = ", ".join(result["grounding_words"]) or "(no shared terms found with the retrieved excerpts)"
    print(f"  established on: {grounding}")
    print("\n" + "-" * 80 + "\n")

Q: Is vitamin C a water-soluble vitamin?
  chunks used: 5 (source(s): source13, source17, source2, source4)
  source pages: [30, 31, 32, 33, 34, 35, 36, 37, 38, 39, 40, 41, 42, 43, 44, 45, 46, 47, 48, 49, 50, 51, 52, 53, 54, 55, 56, 57, 58, 59, 60, 61, 62, 63, 65, 66, 67, 68, 69, 70, 71, 72, 73, 74, 75, 76, 77, 78, 79, 80, 81, 82, 83, 84, 85, 86, 87, 88, 89, 90, 91, 92, 93, 94, 95, 96, 97, 98, 99, 100, 101, 102, 103, 104, 105, 106, 107, 108, 109, 110, 111, 112, 113, 114, 115, 116, 117, 118, 119, 120, 121, 122, 123, 124]
  Short answer: Yes

Short answer: Yes

Excerpt 1 explicitly states that "Ascorbic acid is a water-soluble vitamin commonly known as vitamin C." This is further supported by Excerpt 2, which lists "Vitamin C (ascorbic acid)" under the category of water-soluble vitamins alongside the B-complex vitamins, and Excerpt 3, which states there are 10 water-soluble vitamins and includes "Vitamin C, also known as ascorbic acid" among them.

  established on: states, ascorbic, aci

**Cell #11**

## Summary table

In [6]:
# Cell #12
print(f"{'#':<3} {'short answer':<13} {'chunks':>7} {'pages':>6} {'sources':<12} question")
for i, r in enumerate(results, start=1):
    short = r["short_answer"] or "n/a"
    print(f"{i:<3} {short:<13} {r['chunks_used']:>7} {len(r['source_pages']):>6} "
          f"{','.join(r['source_keys']):<12} {r['question']}")

#   short answer   chunks  pages sources      question
1   Yes                 5     94 source13,source17,source2,source4 Is vitamin C a water-soluble vitamin?
2   No                  5     79 source11,source17,source3 Does eating excess dietary protein get stored directly as body fat?
3   n/a                 5     35 source17,source2,source3 What roles do carbohydrates, proteins, and fats each play in providing energy to the body?
4   No                  5     62 source17,source2 Is saturated fat the only type of dietary fat linked to raised LDL cholesterol?
5   n/a                 5     37 source17,source4,source8 What functions does dietary fiber serve in the digestive system?


**Cell #13**

## Save workspace to GitHub

Synchronize this notebook, answers, and any code changes to GitHub using  (with auto lock recovery and conflict resolution).

In [ ]:
# Cell #14
from reusable_code import save_to_github

save_to_github("stage2_ask_examples1.ipynb - answers verified and synced")
